# VisDrone — yolo26**s**

Lam tron mot size: baseline -> prune 50% -> finetune + CWD -> val.
Ket qua la **hai dong** cua bang: `YOLO26-S` va `Ours-S`.

| | |
|---|---|
| Baseline | `yolo26s.pt` (COCO), 100 epoch tren VisDrone |
| Ours | L1-norm uniform 50% (div 8) + 100 epoch CWD tau=9, kd_layers=neck |
| imgsz / batch / seed | **1024 / 8** / 0 |
| cos_lr / patience / warmup | False / 100 / 3.0 |
| Uoc tinh | ~24h, **3 phien** |

> Bon notebook n/s/m/l dung **y het** cac than so nay. Doi mot cai thoi la ca
> bang het so sanh duoc.
>
> **1024 chu khong phai 640**: khop voi arXiv 2509.12918 (ho cung dung batch 8
> o 1024) de co dong doi dau truc tiep. Bang 640 da chay xong van giu nguyen
> lam bang chinh. Muon quay lai 640 thi doi `IMGSZ` o cell 2, ten run tu doi
> theo nen khong de len nhau.

## Cach chay

1. Settings -> Accelerator **GPU T4 x2**, **Internet: On**
2. **Save & Run All**. Lan dau khong can Add Data.
3. Phien tu dung o **10h** (ket thuc khoang 10.3h, du truoc moc 12h cua Kaggle).
   Cell cuoi bao con thieu bao nhieu epoch -> Add Data
   output cua chinh lan chay nay roi Save & Run All lai.
4. Xong thi gui lai bang 2 dong o cell cuoi.

Logic nam trong `notebooks/share_visdrone/vd_common.py` trong repo, cell setup
tu `git pull` moi lan chay — sua loi o do la ca nhom co ngay, khong phai import
lai notebook.

Repo: https://github.com/xauskeleton/yolo26_prune_cwd

## 1. Setup

In [ ]:
import os
import pathlib
import subprocess
import sys

REPO_DIR = pathlib.Path("/kaggle/working/yolo")
# Luon keo ban moi nhat: logic nam trong repo nen sua o do la ca nhom co ngay.
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/xauskeleton/yolo26_prune_cwd", str(REPO_DIR)], check=True
    )
os.chdir(REPO_DIR)

# Phai dung fork nay, KHONG "pip install ultralytics": checkpoint sau khi prune
# duoc pickle voi ultralytics.nn.tasks_pruned nen ban chinh thuc khong load duoc.
for d in (REPO_DIR, REPO_DIR / "pruning", REPO_DIR / "notebooks" / "share_visdrone"):
    sys.path.insert(0, str(d))
# DDP sinh tien trinh con chay file tam ngoai repo -> phai truyen qua PYTHONPATH.
os.environ["PYTHONPATH"] = str(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=False)

import torch
import vd_common as V

print("GPU:", torch.cuda.device_count(), "| vd_common:", V.__file__)

## 2. Cau hinh

In [ ]:
SIZE = "s"
MODEL = "yolo26s.pt"
RATIO = 0.5

# ====================== DO PHAN GIAI ======================
#  640  = chuan cua nhanh nen model, dung cho bang chinh
#  1024 = khop voi arXiv 2509.12918 (ho cung batch 8) de doi dau truc tiep
IMGSZ = 1024
# ==========================================================

# Batch phai ha theo do phan giai. Do that o 640 batch 16 DDP: 7.3 GB.
# Activation tang theo binh phuong do phan giai -> 1024 batch 16 se ~18.7 GB,
# tran T4 15 GB. Batch 8 con ~9.4 GB, con cho cho teacher forward (chay FP32
# ngoai autocast nen ton them).
BATCH = {640: 16, 896: 12, 1024: 8, 1280: 4}[IMGSZ]

# Ten run kem do phan giai de khong de len ket qua 640 da chay xong. Neu dung
# chung ten, cell resume se keo run 640 da du 100 epoch ve va V.train() bao
# "da du, bo qua" -> khong train gi ca.
SUF = "" if IMGSZ == 640 else f"_{IMGSZ}"
BASE_NAME = "vd_yolo26s" + SUF
OURS_NAME = "vd_ourss" + SUF
PRUNED = REPO_DIR / "weights" / ("yolo26s_vd_pruned50" + SUF + ".pt")

V.init(
    REPO_DIR=REPO_DIR,
    DATA="VisDrone.yaml",  # Ultralytics tu tai 2.3 GB, can Internet: On
    EPOCHS=100,
    BATCH=BATCH,
    IMGSZ=IMGSZ,
    DEVICE="0,1" if torch.cuda.device_count() > 1 else "0",
    # Chot cung thay vi de mac dinh: cac run VOC truoc day khong dong nhat
    # (n/s/l dung batch 32 + cos_lr=True + patience 20-30, m dung 16/False/100).
    COS_LR=False,
    PATIENCE=100,
    WARMUP=3.0,
    # Dung train sau 10 tieng. Dem tu luc BAT DAU TRAIN, khong ke setup.
    # Tong phien: setup ~3p + train 10h + tran mot epoch dang do ~10p
    # (o 1024 moi epoch ~10 phut) + val cuoi ~5p  =  khoang 10.3h.
    # Con gan 2 tieng du truoc moc 12h cua Kaggle.
    STOP_AFTER_H=10.0,
)

# Chi dung khi chi co moi mot file last.pt roi le (phien bi danh dau failed).
# Upload ca thu muc <run_name>/weights/ thi khong can dien gi.
MANUAL_LAST = {BASE_NAME: "", OURS_NAME: ""}

print(BASE_NAME, "|", OURS_NAME, "|", V.CFG["DEVICE"])

## 3. Dataset

In [ ]:
# Dung lai dataset tu output cu neu da Add Data, khong thi de
# Ultralytics tu tai ve /kaggle/temp (khong vao output).
V.setup_dataset()

## 4. Resume

In [ ]:
V.restore(BASE_NAME, OURS_NAME, PRUNED, MANUAL_LAST)

## 5. Baseline yolo26s

In [ ]:
n_base = V.train(BASE_NAME, MODEL)

if n_base < 100:
    print()
    print(f"Baseline moi {n_base}/100 epoch - het gio phien nay.")
    print("Add Data output lan nay roi Save & Run All lai. Cac cell duoi bo qua.")

## 6. Prune 50%

In [ ]:
BEST_BASE = REPO_DIR / "runs" / BASE_NAME / "weights" / "best.pt"

if n_base < 100:
    print("Bo qua: baseline chua xong.")
else:
    V.prune50(SIZE, BEST_BASE, PRUNED, RATIO)

## 7. Finetune + CWD

In [ ]:
if n_base < 100 or not PRUNED.exists():
    print("Bo qua: chua co model da prune.")
    n_ours = 0
else:
    # Teacher la baseline cua chinh size nay.
    n_ours = V.train(
        OURS_NAME,
        str(PRUNED),
        finetune=True,  # build DetectionModelPruned tu maskbndict
        kd=True,
        kd_teacher=str(BEST_BASE),
        kd_method="cwd",
        kd_lambda=0.5,
        kd_layers="neck",
        kd_warmup=5,
        cwd_temperature=9.0,
    )

## 8. Ket qua

In [ ]:
V.report(SIZE, BASE_NAME, OURS_NAME)